# 05.27 - SHAP & Model Interpretability

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Black-box models are powerful but unexplainable. SHAP values explain individual predictions.

## 2. Why Does This Matter?

Explainability is critical for trust, debugging, and compliance. SHAP is the gold standard.

## 3. Prerequisites

- 05.05-05.08: Tree models

## 4. Learning Objectives

- Understand SHAP values conceptually
- Use SHAP with tree models
- Interpret individual predictions
- Create summary plots

## 5. Mental Model

SHAP values = how much each feature pushes the prediction from the baseline.
Positive SHAP = pushes toward class 1.
Negative SHAP = pushes toward class 0.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded.")

## 6. Train Model

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("Accuracy:", round(rf.score(X_test, y_test), 4))

## 7. Install and Use SHAP

In [ ]:
try:
    import shap
    print("SHAP version:", shap.__version__)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap
    print("SHAP installed:", shap.__version__)

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
print("SHAP values shape:", np.array(shap_values).shape)

## 8. Summary Plot

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=wine.feature_names, show=False)
plt.title("SHAP Feature Importance")
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=100, bbox_inches="tight")
plt.show()

## 9. Individual Prediction Explanation

In [ ]:
sample_idx = 0
print("Predicted class:", rf.predict(X_test.iloc[[sample_idx]])[0])
print("True class:", y_test.iloc[sample_idx] if hasattr(y_test, "iloc") else y_test[sample_idx])

plt.figure(figsize=(10, 6))
shap.force_plot(explainer.expected_value[0], shap_values[0][sample_idx], X_test.iloc[sample_idx], feature_names=wine.feature_names, show=False, matplotlib=True)
plt.title("SHAP Force Plot for Sample " + str(sample_idx))
plt.tight_layout()
plt.savefig("shap_force.png", dpi=100, bbox_inches="tight")
plt.show()

## 10. Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SHAP importance
shap_importance = np.abs(shap_values).mean(axis=(0, 2) if len(shap_values[0].shape) > 1 else (0,))
if len(shap_importance) != len(wine.feature_names):
    shap_importance = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    if len(shap_importance) != len(wine.feature_names):
        shap_importance = shap_importance[:len(wine.feature_names)]

axes[0].barh(wine.feature_names, shap_importance)
axes[0].set_title("SHAP Importance")

# RF importance
axes[1].barh(wine.feature_names, rf.feature_importances_)
axes[1].set_title("Random Forest Importance")

plt.tight_layout()
plt.savefig("importance_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## 11. Common Mistakes

1. Using SHAP on non-tree models (slow)
2. Misinterpreting SHAP values
3. Ignoring feature interactions

## 12. Coding Exercises

### Exercise 1: Titanic SHAP
Train RF on Titanic. Explain predictions with SHAP.

### Exercise 2: SHAP Dependence
Create SHAP dependence plots.

In [ ]:
# EXERCISE 1: Titanic SHAP
import seaborn as sns
print("Exercise: SHAP on Titanic dataset.")

In [ ]:
# EXERCISE 2: Dependence plot
print("Exercise: Create SHAP dependence plots.")

## 13. Closed-Book Recall

1. What does a SHAP value represent?
2. How do you interpret a force plot?
3. What is the difference between SHAP and feature importance?

## 14. Teach-Back Questions

Explain SHAP values to a non-technical person. How to use them for debugging.

## 15. Summary

SHAP values explain individual predictions. TreeExplainer is fast for tree models. Summary plots show global importance. Force plots explain individual predictions.

## 16. Further Experiment

1. Try SHAP with linear models.
2. Use SHAP for feature selection.
3. Create a SHAP dashboard.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, pandas, matplotlib, scikit-learn, shap]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```